# Chapter 37: Dependency Injection & Error Handling — Colab Notebook

This notebook actually runs the FastAPI code that the lesson page (`ch37-dependency-injection-error-handling.html`) shows as reference-only. Same reason as Chapter 36: FastAPI needs a real ASGI runtime, which Pyodide can't provide.

Every cell below uses `fastapi.testclient.TestClient` (in-process, no real socket) so it runs identically here or in production.

```
!pip install -q fastapi uvicorn httpx
```


## 37.1 — A Shared Pagination Dependency

In [1]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app = FastAPI(title="Candidate Scoring API")

CANDIDATES = {
    1: {"name": "Alice", "score": 0.92},
    2: {"name": "Bob",   "score": 0.75},
    3: {"name": "Carol", "score": 0.81},
    4: {"name": "Dave",  "score": 0.60},
    5: {"name": "Eve",   "score": 0.55},
}

def get_pagination(skip: int = 0, limit: int = 10):
    return {"skip": skip, "limit": limit}

@app.get("/candidates")
def list_candidates(pagination: dict = Depends(get_pagination)):
    return {"pagination": pagination, "results": list(CANDIDATES.values())[pagination["skip"]:pagination["skip"]+pagination["limit"]]}

client = TestClient(app)
response = client.get("/candidates", params={"skip": 1, "limit": 2})
print("status:", response.status_code)
print("body:", response.json())


status: 200
body: {'pagination': {'skip': 1, 'limit': 2}, 'results': [{'name': 'Bob', 'score': 0.75}, {'name': 'Carol', 'score': 0.81}]}


## 37.2 — Class-Based Dependencies

In [2]:
class CommonQueryParams:
    def __init__(self, q: str | None = None, skip: int = 0, limit: int = 10):
        self.q = q
        self.skip = skip
        self.limit = limit

@app.get("/candidates/search")
def search_candidates(commons: CommonQueryParams = Depends(CommonQueryParams)):
    return {"q": commons.q, "skip": commons.skip, "limit": commons.limit}

client = TestClient(app)
response = client.get("/candidates/search", params={"q": "alice", "limit": 3})
print("status:", response.status_code)
print("body:", response.json())


status: 200
body: {'q': 'alice', 'skip': 0, 'limit': 3}


## 37.3 — HTTPException

In [3]:
from fastapi import HTTPException

@app.get("/candidates/{candidate_id}")
def get_candidate(candidate_id: int):
    if candidate_id not in CANDIDATES:
        raise HTTPException(status_code=404, detail="Candidate not found")
    return CANDIDATES[candidate_id]

client = TestClient(app)

response = client.get("/candidates/1")
print("GET /candidates/1 ->", response.status_code, response.json())

response = client.get("/candidates/999")
print("GET /candidates/999 ->", response.status_code, response.json())


GET /candidates/1 -> 200 {'name': 'Alice', 'score': 0.92}
GET /candidates/999 -> 404 {'detail': 'Candidate not found'}


## 37.4 — Custom Exception Classes & Handlers

In [4]:
from fastapi import Request
from fastapi.responses import JSONResponse

# Exception handlers must be registered BEFORE an app serves its first request
# (Starlette builds and caches its handler stack on the first call). The `app` above has
# already handled requests, so this section builds a fresh app to keep the demo honest.
handler_app = FastAPI()

class CandidateNotFoundError(Exception):
    def __init__(self, candidate_id: int):
        self.candidate_id = candidate_id

@handler_app.exception_handler(CandidateNotFoundError)
def candidate_not_found_handler(request: Request, exc: CandidateNotFoundError):
    return JSONResponse(
        status_code=404,
        content={"error": f"Candidate {exc.candidate_id} not found"},
    )

@handler_app.get("/candidates-v2/{candidate_id}")
def get_candidate_v2(candidate_id: int):
    if candidate_id not in CANDIDATES:
        raise CandidateNotFoundError(candidate_id)
    return CANDIDATES[candidate_id]

handler_client = TestClient(handler_app, raise_server_exceptions=False)
response = handler_client.get("/candidates-v2/999")
print("status:", response.status_code)
print("body:", response.json())


status: 404
body: {'error': 'Candidate 999 not found'}


## 37.5 — Sub-Dependencies

In [5]:
def get_sorted_pagination(pagination: dict = Depends(get_pagination), sort_by: str = "score"):
    return {**pagination, "sort_by": sort_by}

@app.get("/candidates-sorted")
def list_sorted_candidates(query: dict = Depends(get_sorted_pagination)):
    return query

client = TestClient(app)
response = client.get("/candidates-sorted", params={"skip": 0, "limit": 5, "sort_by": "name"})
print("status:", response.status_code)
print("body:", response.json())


status: 200
body: {'skip': 0, 'limit': 5, 'sort_by': 'name'}


## 37.6 — Status Codes In Practice

In [6]:
@app.post("/candidates", status_code=201)
def create_candidate(name: str, score: float):
    new_id = max(CANDIDATES) + 1
    CANDIDATES[new_id] = {"name": name, "score": score}
    return {"id": new_id, "name": name, "score": score}

@app.delete("/candidates/{candidate_id}", status_code=204)
def delete_candidate(candidate_id: int):
    if candidate_id not in CANDIDATES:
        raise HTTPException(status_code=404, detail="Candidate not found")
    del CANDIDATES[candidate_id]
    return None

client = TestClient(app)
r1 = client.post("/candidates", params={"name": "Grace", "score": 0.81})
print("POST ->", r1.status_code, r1.json())

r2 = client.delete(f"/candidates/{r1.json()['id']}")
print("DELETE ->", r2.status_code, repr(r2.text))


POST -> 201 {'id': 6, 'name': 'Grace', 'score': 0.81}
DELETE -> 204 ''


## 37.7 — Testing With Dependency Overrides

In [7]:
def override_get_pagination():
    return {"skip": 0, "limit": 2}

app.dependency_overrides[get_pagination] = override_get_pagination
client = TestClient(app)

response = client.get("/candidates")
assert response.status_code == 200
assert response.json()["pagination"]["limit"] == 2
print("Override in effect:", response.json()["pagination"])

app.dependency_overrides.clear()

# Prove the override is really gone
response = client.get("/candidates")
print("After clearing override:", response.json()["pagination"])


Override in effect: {'skip': 0, 'limit': 2}
After clearing override: {'skip': 0, 'limit': 10}


## Mini Project: Candidate Scoring API — With Shared Lookup & Errors

Extends Chapter 36's API with one shared `get_candidate_or_404` dependency reused by GET, PUT, and DELETE, plus a registered exception handler.

In [8]:
from fastapi import FastAPI, Depends, HTTPException, Request
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

project_app = FastAPI(title="Candidate Scoring API — Mini Project 37")

PROJECT_CANDIDATES = {
    1: {"name": "Alice", "score": 0.92},
    2: {"name": "Bob",   "score": 0.75},
}

class CandidateUpdate(BaseModel):
    name: str
    score: float = Field(gt=0, le=1)

class CandidateOut(BaseModel):
    id: int
    name: str
    score: float

class CandidateNotFoundError(Exception):
    def __init__(self, candidate_id: int):
        self.candidate_id = candidate_id

@project_app.exception_handler(CandidateNotFoundError)
def not_found_handler(request: Request, exc: CandidateNotFoundError):
    return JSONResponse(status_code=404, content={"error": f"Candidate {exc.candidate_id} not found"})

def get_candidate_or_404(candidate_id: int) -> dict:
    if candidate_id not in PROJECT_CANDIDATES:
        raise CandidateNotFoundError(candidate_id)
    return PROJECT_CANDIDATES[candidate_id]

@project_app.get("/candidates/{candidate_id}", response_model=CandidateOut)
def get_candidate(candidate_id: int, candidate: dict = Depends(get_candidate_or_404)):
    return {"id": candidate_id, **candidate}

@project_app.put("/candidates/{candidate_id}", response_model=CandidateOut)
def update_candidate(candidate_id: int, body: CandidateUpdate, _existing: dict = Depends(get_candidate_or_404)):
    PROJECT_CANDIDATES[candidate_id] = body.model_dump()
    return {"id": candidate_id, **body.model_dump()}

@project_app.delete("/candidates/{candidate_id}", status_code=204)
def delete_candidate(candidate_id: int, _existing: dict = Depends(get_candidate_or_404)):
    del PROJECT_CANDIDATES[candidate_id]
    return None

client = TestClient(project_app, raise_server_exceptions=False)

r1 = client.get("/candidates/1")
assert r1.status_code == 200 and r1.json()["name"] == "Alice"

r2 = client.get("/candidates/999")
assert r2.status_code == 404

r3 = client.put("/candidates/2", json={"name": "Bob", "score": 0.80})
assert r3.status_code == 200 and r3.json()["score"] == 0.80

r4 = client.delete("/candidates/2")
assert r4.status_code == 204

r5 = client.get("/candidates/2")
assert r5.status_code == 404

print("Project checklist: PASSED (shared dependency, PUT, DELETE, custom exception handler, clean 404s)")


Project checklist: PASSED (shared dependency, PUT, DELETE, custom exception handler, clean 404s)


### Next: Chapter 38 — Databases with SQLAlchemy (Colab)